# hipBLASLt Offline GEMM Tuning Demo
**hipBLASLt** is a library that provides **General Matrix-Matrix (GEMM)** operations with flexible APIs and extends functionality beyond the traditional BLAS library. To learn more, [What is hipBLASLt?](https://rocm.docs.amd.com/projects/hipBLASLt/en/latest/what-is-hipBLASLt.html)

This demo provide a guide how to leverage hipBLASLt offline gemm tuning method to optimize GEMM kernels used in `Qwen3-32B` model deployment with `vLLM` Framework.

**Pipeline:** build clients (optional) → capture **hipblaslt-bench** lines from inference → **`gemm_tuning.py`** → **`tuning_analysis.py`** → apply **`tuning.txt`** at inference (`HIPBLASLT_TUNING_OVERRIDE_FILE`).

Reference: 
- [AMD blog — hipBLASLt offline tuning](https://rocm.blogs.amd.com/artificial-intelligence/hipblaslt_offline_tuning/README.html)
- [hipBLASLt offline tuning guide](https://github.com/ROCm/rocm-libraries/blob/develop/projects/hipblaslt/utilities/QuickTune/README.md)


## Prerequisites

- **ROCm**
- **hipblaslt-bench** on `PATH` (typically under `hipblaslt/build/release/clients`).
- **Python:**, **pandas**; **ipywidgets**


In [22]:
%pip install "pandas" "ipywidgets"


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
import os
import shutil
import subprocess
from pathlib import Path
from typing import Tuple


def find_repo_roots() -> Tuple[Path, Path]:
    """Return (REPO_ROOT, QUICKTUNE) for common layouts.

    Primary: ``force_demo/hipblaslt_demo`` after ``prepare_env.sh`` — this notebook and
    ``hipblaslt_offline_tuning.md`` sit next to ``rocm-libraries/`` (``REPO_ROOT`` = cwd).

    Legacy: repo root with ``offline/hipblaslt_offline_tuning.md``; or walk parents.
    """
    here = Path.cwd().resolve()
    quick = lambda r: r / "rocm-libraries" / "projects" / "hipblaslt" / "utilities" / "QuickTune"

    # prepare_env.sh: rocm-libraries cloned under hipblaslt_demo/ (same dir as this notebook)
    if (here / "hipblaslt_offline_tuning.md").is_file() and (here / "rocm-libraries").is_dir():
        repo = here
        return repo, quick(repo)

    # hipblaslt-tuning style: repo root contains offline/hipblaslt_offline_tuning.md
    if (here / "offline" / "hipblaslt_offline_tuning.md").is_file() and (here / "rocm-libraries").is_dir():
        repo = here
        return repo, quick(repo)

    # Doc in cwd, rocm-libraries one level up (older layouts)
    if (here / "hipblaslt_offline_tuning.md").is_file() and (here.parent / "rocm-libraries").is_dir():
        repo = here.parent
        return repo, quick(repo)

    for p in here.parents:
        if (p / "offline" / "hipblaslt_offline_tuning.md").is_file() and (
            p / "rocm-libraries" / "projects" / "hipblaslt" / "utilities" / "QuickTune" / "gemm_tuning.py"
        ).is_file():
            return p, quick(p)

    raise FileNotFoundError(
        "Could not find rocm-libraries + QuickTune. "
        "cd into force_demo/hipblaslt_demo (next to prepare_env.sh), run bash prepare_env.sh, "
        "then set kernel cwd to hipblaslt_demo and re-run this cell."
    )


REPO_ROOT, QUICKTUNE = find_repo_roots()
OFFLINE_DIR = REPO_ROOT / "offline"
BENCH_DIR = REPO_ROOT / "rocm-libraries" / "projects" / "hipblaslt" / "build" / "release" / "clients"
OUTPUT_PATH = QUICKTUNE / "offline_tuning_result"
EXAMPLE_LOG = QUICKTUNE / "example" / "Qwen3-32B_hipblaslt.log"

os.environ["PATH"] = str(BENCH_DIR) + os.pathsep + os.environ.get("PATH", "")
print("REPO_ROOT:", REPO_ROOT)
print("QUICKTUNE:", QUICKTUNE)
print("BENCH_DIR:", BENCH_DIR)
print("hipblaslt-bench:", shutil.which("hipblaslt-bench") or "(not on PATH — run prepare_env.sh or fix cwd)")

REPO_ROOT: /root/workspace/bytedance/demo/force_demo/hipblaslt_demo
QUICKTUNE: /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/utilities/QuickTune
BENCH_DIR: /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/build/release/clients
hipblaslt-bench: /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/build/release/clients/hipblaslt-bench


## Step 0: (Optional) Prepare environment

Build **hipblaslt-bench** (and other hipBLASLt clients) for GEMM evaluation during tuning.

From a **terminal** (not this notebook kernel), run the demo helper **[`prepare_env.sh`](prepare_env.sh)** in the **`hipblaslt_demo/`** folder (same directory as this notebook). It clones `rocm-libraries` if needed, installs GTest/GMock, and runs `install.sh`.

```bash
cd /path/to/force_demo/hipblaslt_demo
export GPU_ARCH=gfx942          # optional; default in script is gfx942
export ROCM_BRANCH=release/rocm-rel-7.2   # optional
bash prepare_env.sh
```

Use the **`export PATH=...`** line the script prints, then **restart the kernel** and re-run the setup cell below so **`hipblaslt-bench`** is on `PATH` (the code cell prepends `build/release/clients` under your detected `REPO_ROOT`).

Full options and defaults: see **[`hipblaslt_offline_tuning.md`](hipblaslt_offline_tuning.md)** § Step 0.

In [24]:
# Optional: verify bench binary exists on disk even if not yet on PATH
bench = BENCH_DIR / "hipblaslt-bench"
print("hipblaslt-bench:", bench)
print("exists:", bench.is_file(), "(if False, run Step 0 build)")

hipblaslt-bench: /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/build/release/clients/hipblaslt-bench
exists: True (if False, run Step 0 build)


## Step 1: Collect hipBLASLt log from inference

`HIPBLASLT_LOG_MASK=32` emits **hipblaslt-bench–style** lines into `HIPBLASLT_LOG_FILE`. Replace **`run_model.py`** with your driver.

Set **`RUN_CAPTURE`** only if you have a real script (can be long / needs GPU).

In [25]:
RUN_CAPTURE = False  # set True and set CAPTURE_CMD to your inference command
LOG_FILE = REPO_ROOT / "offline" / "Qwen3-32B_hipblaslt.log"
CAPTURE_CMD = ["python3", "run_model.py"]  # change to your entrypoint

os.environ["HIPBLASLT_LOG_MASK"] = "32"
os.environ["HIPBLASLT_LOG_FILE"] = str(LOG_FILE)
# os.environ["HIP_VISIBLE_DEVICES"] = "0"

if RUN_CAPTURE:
    subprocess.run(CAPTURE_CMD, cwd=REPO_ROOT, check=True)
    print("Wrote", LOG_FILE)
else:
    print("Skipping capture. Example log for tuning:", EXAMPLE_LOG, "exists:", EXAMPLE_LOG.is_file())

Skipping capture. Example log for tuning: /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/utilities/QuickTune/example/Qwen3-32B_hipblaslt.log exists: True


## Step 2: Run GEMM offline tuning (`gemm_tuning.py`)

Runs from **`QUICKTUNE`**. Uses bundled **`example/Qwen3-32B_hipblaslt.log`** by default. **GPU-heavy** — set **RUN_GEMM_TUNING** to **True** in the next cell’s dropdown, then **re-run that same cell** to execute (the dropdown is created **once** per kernel session so your choice is not reset on each run).


In [27]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    display = None

REQUESTED_SOLUTION = 128
os.environ["HIPBLASLT_LOG_MASK"] = "32"

# Create the dropdown once. Re-running this cell would reset `value=False` if we
# constructed a new Dropdown every time — so reuse the same widget and only read
# `.value` when you re-run after changing the UI.
run_gemm_tuning_dd = widgets.Dropdown(
    options=[False, True],
    value=False,
    description="RUN_GEMM_TUNING:",
    disabled=False,
    style={"description_width": "max-content"},
    layout=widgets.Layout(width="520px", min_width="360px", min_height="38px"),
)
display(run_gemm_tuning_dd)


Dropdown(description='RUN_GEMM_TUNING:', layout=Layout(min_height='38px', min_width='360px', width='520px'), o…

In [29]:
RUN_GEMM_TUNING = run_gemm_tuning_dd.value
if RUN_GEMM_TUNING:
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python3",
        str(QUICKTUNE / "gemm_tuning.py"),
        "--input_file",
        str(EXAMPLE_LOG),
        "--output_path",
        str(OUTPUT_PATH),
        "--requested_solution",
        str(REQUESTED_SOLUTION),
        "--swizzleA",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=QUICKTUNE, check=True)
else:
    print(
        "RUN_GEMM_TUNING is False — select True in the dropdown, then re-run **this** cell "
        "(the widget is not recreated, so your choice is kept)."
    )
    print("Expected artifacts under:", OUTPUT_PATH)


Running: python3 /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/utilities/QuickTune/gemm_tuning.py --input_file /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/utilities/QuickTune/example/Qwen3-32B_hipblaslt.log --output_path /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/utilities/QuickTune/offline_tuning_result --requested_solution 128 --swizzleA
parsing time elapsed = 0.00046133995056152344 seconds
After removing duplicate, the hipblaslt log is saved at
 /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/utilities/QuickTune/offline_tuning_result/unique_Qwen3-32B_hipblaslt.log 

hipBLASLt version: 100202
hipBLASLt git version: dabb6df2b9
Query device success: there are 1 devices. (Target device ID is 0)
Device ID 0 :  gfx942:sramecc+:xnack-
with 206.1 GB memory, max. SCLK 1850 MHz, max. MCLK 1300 MHz, compute capabi

## Step 3: Analyze results (`tuning_analysis.py`)

Runs if **`tuning_result.csv`** exists (after Step 2).

In [30]:
# Matches remove_duplicate.parse_input_log: unique_<basename of --input_file>
unique_log = OUTPUT_PATH / f"unique_{EXAMPLE_LOG.name}"
tuning_csv = OUTPUT_PATH / "tuning_result.csv"
analysis_csv = OUTPUT_PATH / "analysis.csv"

if unique_log.is_file() and tuning_csv.is_file():
    cmd = [
        "python3",
        str(QUICKTUNE / "tuning_analysis.py"),
        "--input_log",
        str(unique_log),
        "--input_csv",
        str(tuning_csv),
        "--output_csv",
        str(analysis_csv),
    ]
    proc = subprocess.run(cmd, cwd=QUICKTUNE, capture_output=True, text=True)
    print(proc.stdout)
    if proc.stderr:
        print("stderr:", proc.stderr)
    proc.check_returncode()
    print("Wrote:", analysis_csv)
else:
    print("Missing inputs; run Step 2 first.")
    print(" expected:", unique_log, tuning_csv)

Total -m -n -k combos: 13

                  (-m, -n, -k)   count  baseline/tuned  total_baseline(us)   total_tuned(us)   baseline%    tuned%
------------------------------------------------------------------------------------------------------------------
            (6400, 4000, 5120)       1       107.79%            1072.270           994.773       45.24%     48.92%
            (5120, 4000, 3200)       1        123.5%             604.450           489.420       25.50%     24.07%
            (5120, 4000, 1024)       1        139.6%             235.611           168.772        9.94%      8.30%
            (1280, 4000, 5120)       1       107.76%             222.372           206.364        9.38%     10.15%
              (18992, 1, 5120)       1       133.68%              76.833            57.475        3.24%      2.83%
               (6400, 8, 5120)       1       163.29%              34.901            21.373        1.47%      1.05%
               (6400, 1, 5120)       1        158.2% 

## Step 4: Adopt tuned solution at inference

**`--swizzleA`** changes the bench command (e.g. **`--transA T`**). Your integration must match that layout (weight transpose / preshuffle as needed).

Use **`HIPBLASLT_TUNING_OVERRIDE_FILE`** so you do not overwrite a dev **`HIPBLASLT_TUNING_FILE`**:

```bash
unset HIPBLASLT_TUNING_FILE
export HIPBLASLT_TUNING_OVERRIDE_FILE="${QUICKTUNE}/offline_tuning_result/tuning.txt"
python3 run_model.py
```

The next cell sets the override env var to the path from this notebook’s **`OUTPUT_PATH`** (only if **`tuning.txt`** exists).

In [17]:
tuning_txt = OUTPUT_PATH / "tuning.txt"
if tuning_txt.is_file():
    os.environ.pop("HIPBLASLT_TUNING_FILE", None)
    os.environ["HIPBLASLT_TUNING_OVERRIDE_FILE"] = str(tuning_txt)
    print("HIPBLASLT_TUNING_OVERRIDE_FILE=", os.environ["HIPBLASLT_TUNING_OVERRIDE_FILE"])
    print("Run your inference in this kernel/session, or export the same in your shell.")
else:
    print("No tuning.txt yet — complete Step 2 first:", tuning_txt)

HIPBLASLT_TUNING_OVERRIDE_FILE= /root/workspace/bytedance/demo/force_demo/hipblaslt_demo/rocm-libraries/projects/hipblaslt/utilities/QuickTune/offline_tuning_result/tuning.txt
Run your inference in this kernel/session, or export the same in your shell.


### Optional: peek at CSV outputs

In [18]:
tuning_csv = OUTPUT_PATH / "tuning_result.csv"
analysis_csv = OUTPUT_PATH / "analysis.csv"

try:
    import pandas as pd
except ImportError:
    pd = None
    print("Install pandas to preview CSVs: pip install pandas")

if pd is not None:
    for name, path in [
        ("tuning_result.csv", tuning_csv),
        ("analysis.csv", analysis_csv),
    ]:
        if path.is_file():
            df = pd.read_csv(path)
            print(name, "shape", df.shape)
            display(df.head(12))
        else:
            print("Missing:", path)

tuning_result.csv shape (13, 17)


,m,n,k,lda,ldb,ldc,ldd,a_type,b_type,c_type,d_type,count,baseline_latency(us),baseline_solution_idx,tuned_latency(us),tuned_solution_idx,baseline/tuned
0,1280,1,5120,1280,5120,1280,1280,bf16_r,bf16_r,bf16_r,bf16_r,5,14.5622,190782,11.9871,202916,121.48%
1,5120,1,1024,5120,1024,5120,5120,bf16_r,bf16_r,bf16_r,bf16_r,5,11.4335,190782,11.9778,202486,95.46%
2,6400,1,5120,6400,5120,6400,6400,bf16_r,bf16_r,bf16_r,bf16_r,5,35.3285,190783,21.2354,203144,166.37%
3,5120,1,3200,5120,3200,5120,5120,bf16_r,bf16_r,bf16_r,bf16_r,5,16.9614,190782,13.6257,202486,124.48%
4,18992,1,5120,5120,5120,18992,18992,bf16_r,bf16_r,bf16_r,bf16_r,5,76.3927,197018,57.3914,203605,133.11%
5,1280,8,5120,1280,5120,1280,1280,bf16_r,bf16_r,bf16_r,bf16_r,5,15.7223,190782,10.5898,202916,148.47%
6,5120,8,1024,5120,1024,5120,5120,bf16_r,bf16_r,bf16_r,bf16_r,5,12.7221,190782,10.6322,203122,119.66%
7,6400,8,5120,6400,5120,6400,6400,bf16_r,bf16_r,bf16_r,bf16_r,5,34.9816,190783,21.6799,203144,161.35%
8,5120,8,3200,5120,3200,5120,5120,bf16_r,bf16_r,bf16_r,bf16_r,5,17.0521,190782,13.6791,202486,124.66%
9,1280,4000,5120,1280,5120,1280,1280,bf16_r,bf16_r,bf16_r,bf16_r,5,221.1050,191184,206.5670,203376,107.04%


analysis.csv shape (14, 13)


,m,n,k,count,baseline_latency(us),baseline_solution_idx,tuned_latency(us),tuned_solution_idx,baseline/tuned,total_baseline_time(us),total_tuned_time(us),baseline_time_percent,tuned_time_percent
0,6400,4000,5120,1,1060.75,191184,992.253,202662,106.9%,1060.750,992.253,45.00,48.78
1,5120,4000,3200,1,604.649,191196,492.693,203062,122.72%,604.649,492.693,25.65,24.22
2,5120,4000,1024,1,235.647,191190,169.713,202667,138.85%,235.647,169.713,10.00,8.34
3,1280,4000,5120,1,221.105,191184,206.567,203376,107.04%,221.105,206.567,9.38,10.16
4,18992,1,5120,1,76.3927,197018,57.3914,203605,133.11%,76.393,57.391,3.24,2.82
5,6400,1,5120,1,35.3285,190783,21.2354,203144,166.37%,35.328,21.235,1.50,1.04
6,6400,8,5120,1,34.9816,190783,21.6799,203144,161.35%,34.982,21.680,1.48,1.07
7,5120,8,3200,1,17.0521,190782,13.6791,202486,124.66%,17.052,13.679,0.72,0.67
8,5120,1,3200,1,16.9614,190782,13.6257,202486,124.48%,16.961,13.626,0.72,0.67
9,1280,8,5120,1,15.7223,190782,10.5898,202916,148.47%,15.722,10.590,0.67,0.52


## Cleanup session

Deletes the **`offline_tuning_result`** directory under **`QUICKTUNE`** so the next **`gemm_tuning.py`** run gets a clean output tree (no stale **`tuning_result.csv`**, **`tuning.txt`**, reproduce logs, etc.).

**Destructive:** use the **RUN_CLEANUP_OFFLINE_RESULT** dropdown in the next cell (**True** deletes the tree). Re-run that cell after choosing **True**. The dropdown is created **once** per kernel session so the value is not reset on each run. If **`HIPBLASLT_TUNING_OVERRIDE_FILE`** pointed inside that folder, it is cleared after a successful cleanup.


In [19]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    display = None

run_cleanup_offline_dd = widgets.Dropdown(
    options=[False, True],
    value=False,
    description="RUN_CLEANUP_OFFLINE_RESULT:",
    disabled=False,
    style={"description_width": "max-content"},
    layout=widgets.Layout(width="520px", min_width="360px", min_height="38px"),
)
display(run_cleanup_offline_dd)

Dropdown(description='RUN_CLEANUP_OFFLINE_RESULT:', layout=Layout(min_height='38px', min_width='360px', width=…

In [21]:
def _cleanup_offline_tuning_result() -> None:
    outp = OUTPUT_PATH.resolve()
    qt = QUICKTUNE.resolve()
    if outp.name != "offline_tuning_result":
        raise RuntimeError(f"Refusing cleanup: folder name must be offline_tuning_result, got {outp.name!r}")
    if outp.parent != qt:
        raise RuntimeError(f"Refusing cleanup: {outp} is not directly under QUICKTUNE {qt}")

    override = os.environ.get("HIPBLASLT_TUNING_OVERRIDE_FILE")
    if outp.is_dir():
        shutil.rmtree(outp)
        print("Removed:", outp)
    else:
        print("Nothing to remove (missing):", outp)

    if override:
        try:
            op = Path(override).expanduser().resolve()
            if op == outp or outp in op.parents:
                os.environ.pop("HIPBLASLT_TUNING_OVERRIDE_FILE", None)
                print("Cleared HIPBLASLT_TUNING_OVERRIDE_FILE (pointed under removed tree).")
        except OSError:
            pass

RUN_CLEANUP_OFFLINE_RESULT = run_cleanup_offline_dd.value
if RUN_CLEANUP_OFFLINE_RESULT:
    _cleanup_offline_tuning_result()
else:
    print(
        "RUN_CLEANUP_OFFLINE_RESULT is False — select True in the dropdown, then re-run **this** cell to delete "
        "(the widget is not recreated, so your choice is kept)."
    )

RUN_CLEANUP_OFFLINE_RESULT is False — select True in the dropdown, then re-run **this** cell to delete (the widget is not recreated, so your choice is kept).
